# 01 · Data Acquisition — Dhaka PM2.5

Builds the modeling master: OpenAQ ground-truth **PM2.5** + ERA5 **reanalysis** meteorology + NASA FIRMS fires + calendar proxies, validated against `data/reference/dhaka_aq_master.csv`.

> **Run top-to-bottom only** (Restart kernel and run all). Out-of-order execution is the #1 cause of irreproducible notebook results.

Target = **PM2.5** (no CO₂; CO/NO₂/SO₂ are covariates). Met is ERA5 **reanalysis**. No measured traffic — activity = calendar + FIRMS.

In [ ]:
# === Bootstrap (local Jupyter / Antigravity IDE) ===
import os, sys, random
from pathlib import Path
import numpy as np

SEED = 42
random.seed(SEED); np.random.seed(SEED)

def _find_root(marker="config/sources.yaml"):
    here = Path.cwd().resolve()
    for cand in [here, *here.parents]:
        if (cand / marker).exists():
            return cand
    raise FileNotFoundError(
        "Couldn't find config/sources.yaml. Open/run this notebook from inside the "
        "dhaka-pm25-forecast project folder."
    )

PROJECT_ROOT = _find_root()
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))
for d in ["data/raw","data/interim","data/processed","logs","reports/figures","models"]:
    os.makedirs(d, exist_ok=True)
print("PROJECT_ROOT:", PROJECT_ROOT)

In [ ]:
# === API keys (.env) ===
from dotenv import load_dotenv
load_dotenv()  # reads .env in the project root
OPENAQ_API_KEY = os.getenv("OPENAQ_API_KEY")
FIRMS_MAP_KEY  = os.getenv("FIRMS_MAP_KEY")
assert OPENAQ_API_KEY, "Add OPENAQ_API_KEY to your .env file."
print("OpenAQ key:", bool(OPENAQ_API_KEY), "| FIRMS key:", bool(FIRMS_MAP_KEY))

In [ ]:
import yaml, pandas as pd
with open("config/sources.yaml") as f:
    CFG = yaml.safe_load(f)
PROJ, DR = CFG["project"], CFG["date_range"]
print("City:", PROJ["city"], "| range:", DR["start"], "->", DR["end"])
print("Reference target:", CFG["paths"]["reference"])

## Step 0 · Station auto-discovery
Queries OpenAQ v3 for every station within the configured radius of Dhaka and keeps the PM2.5 reporters. No manual IDs. Result is written to `config/stations_resolved.yaml` for reproducibility.

In [ ]:
import requests

def discover_stations(cfg, api_key):
    p, disc = cfg["project"], cfg["openaq"]["discovery"]
    r = requests.get(
        "https://api.openaq.org/v3/locations",
        headers={"X-API-Key": api_key},
        params={"coordinates": f"{p['lat']},{p['lon']}",
                "radius": disc["radius_m"], "limit": 100},
        timeout=30,
    )
    r.raise_for_status()
    results = r.json().get("results", [])
    rows = []
    for loc in results:
        try:  # v3 nests params under sensors; guard in case the shape differs
            params_rep = sorted({s["parameter"]["name"] for s in loc.get("sensors", [])})
        except Exception:
            params_rep = []
        c = loc.get("coordinates", {}) or {}
        rows.append({"id": loc.get("id"), "name": loc.get("name"),
                     "lat": c.get("latitude"), "lon": c.get("longitude"),
                     "parameters": params_rep})
    return pd.DataFrame(rows)

stations = discover_stations(CFG, OPENAQ_API_KEY)
print(f"Found {len(stations)} stations within {CFG['openaq']['discovery']['radius_m']/1000:.0f} km")
display(stations)

req = CFG["openaq"]["discovery"]["require_parameter"]
pm = stations[stations["parameters"].apply(lambda ps: req in ps)] if len(stations) else stations
with open("config/stations_resolved.yaml", "w") as f:
    yaml.safe_dump({"location_ids": pm["id"].tolist(),
                    "stations": pm.to_dict("records")}, f, sort_keys=False)
print(f"{len(pm)} PM2.5 stations -> config/stations_resolved.yaml")
print("MULTI-STATION (good for journal tier)" if len(pm) > 1 else "SINGLE-STATION (tier ceiling)")

## Step 1 · OpenAQ hourly ground truth

In [ ]:
# === Step 1 · OpenAQ v3 hourly ground truth ===
# Fetches pm25 (TARGET) + pm10/o3/co/no2/so2 covariates for the REFERENCE-GRADE
# stations, paginated, cached to data/raw/openaq/, with retry/backoff.
from src.ingest.openaq import fetch_openaq_hourly

# Station ids come from config/stations_resolved.yaml (written by Step 0).
with open("config/stations_resolved.yaml") as f:
    resolved = yaml.safe_load(f) or {}
loc_ids = resolved.get("location_ids") or []
if not loc_ids and "pm" in dir() and len(pm):
    loc_ids = pm["id"].tolist()          # fallback to in-memory discovery
assert loc_ids, "No resolved PM2.5 stations. Re-run Step 0 (auto-discovery)."

# Ground-truth = REFERENCE-GRADE / fixed-site monitors only. Auto-discovery also
# returns ~18 scattered low-cost (Smart Air) units that are uncalibrated and only
# start ~2026; averaging them into openaq_pm25 skews the median and risks the 15%
# drift gate. We keep the monitors whose sensors reconstruct the continuous
# reference record 2016-2025:
#   22      = SPARTAN - Dhaka University   (pm25 sensor 41 - currently empty)
#   2445    = US Diplomatic Post: Dhaka    (pm25 sensor 4680: 2016-03..2016-11)
#   8415    = Dhaka (regulatory)           (pm25 sensor 24434: 2016-11..2025-03; also o3)
#   3194367 = Jahangirnagar Univ. monitor  (pm25 sensor 11106529: 2024-10..2026-05)
# 24434 is the spine; 11106529 fills 2025-2026 and tracks 24434 within ~4% in the
# Jan-Mar 2025 overlap, so it does not distort the distribution gate.
REFERENCE_GRADE_IDS = [22, 2445, 8415, 3194367]
used_ids = [i for i in REFERENCE_GRADE_IDS if i in loc_ids]
if not used_ids:                          # discovery shape changed -> don't silently drop everything
    print("WARNING: no reference-grade ids in resolved set; falling back to ALL stations.")
    used_ids = loc_ids
print(f"Reference-grade target stations: {used_ids} (of {len(loc_ids)} discovered)")

stations_df = pd.DataFrame({"id": used_ids})
aq_hourly = fetch_openaq_hourly(
    stations_df,
    start=DR["start"], end=DR["end"],
    api_key=OPENAQ_API_KEY,
    parameters=CFG["openaq"]["parameters"],
)
print("aq_hourly:", aq_hourly.shape, "| cols:", list(aq_hourly.columns))
print("UTC span:", aq_hourly.index.min(), "->", aq_hourly.index.max())
print("PM2.5 non-null hours:", int(aq_hourly["openaq_pm25"].notna().sum())
      if "openaq_pm25" in aq_hourly else "NO PM2.5 — check stations")
aq_hourly.head()

## Step 2 · Open-Meteo ERA5 reanalysis 

In [ ]:
# === Step 2 · Open-Meteo ERA5 reanalysis ===
# ERA5 REANALYSIS at the centroid (NOT station obs). Ten hourly vars from config,
# chunked by year, cached to data/raw/open_meteo/.
from src.ingest.open_meteo import fetch_era5_hourly

met_hourly = fetch_era5_hourly(
    lat=PROJ["lat"], lon=PROJ["lon"],
    start=DR["start"], end=DR["end"],
    hourly_vars=CFG["open_meteo"]["hourly"],
    endpoint=CFG["open_meteo"]["archive_endpoint"],
)
print("met_hourly (ERA5 reanalysis):", met_hourly.shape)
print("cols:", list(met_hourly.columns))
print("UTC span:", met_hourly.index.min(), "->", met_hourly.index.max())
met_hourly.head()

## Step 3 · NASA FIRMS fire activity — *completed by PROMPT_1*

In [ ]:
# === Step 3 · NASA FIRMS fire activity ===
# Daily active-fire count + summed FRP within firms.radius_km of the centroid.
# Paged by 5-day windows (area CSV API caps at 5); older windows use the _SP
# archive product. Cached to data/raw/firms/. Skips gracefully (zeros) if
# FIRMS_MAP_KEY is absent.
from src.ingest.firms import fetch_fire_daily

if FIRMS_MAP_KEY:
    fire_daily = fetch_fire_daily(
        lat=PROJ["lat"], lon=PROJ["lon"],
        radius_km=CFG["firms"]["radius_km"],
        start=DR["start"], end=DR["end"],
        map_key=FIRMS_MAP_KEY,
        nrt_source=CFG["firms"]["source"],
    )
else:
    print("WARNING: FIRMS_MAP_KEY missing -> fire columns set to 0 (non-fatal).")
    _idx = pd.date_range(DR["start"], pd.Timestamp(DR["end"]) - pd.Timedelta(days=1), freq="D")
    fire_daily = pd.DataFrame({"firms_count": 0.0, "firms_frp_sum": 0.0}, index=_idx)

print("fire_daily:", fire_daily.shape, "| total detections:", int(fire_daily["firms_count"].sum()))
fire_daily.describe()

## Step 4 · Calendar proxies  *(pure — no API, already implemented)*

In [ ]:
def add_calendar(idx_utc):
    local = pd.DatetimeIndex(idx_utc).tz_convert("Asia/Dhaka")
    cal = pd.DataFrame(index=idx_utc)
    cal["hour"]         = local.hour
    cal["day_of_week"]  = local.dayofweek
    cal["month"]        = local.month
    cal["is_weekend"]   = local.dayofweek.isin([4, 5]).astype(int)   # Fri/Sat in Bangladesh
    cal["is_rush_hour"] = local.hour.isin([8,9,10,17,18,19]).astype(int)
    cal["hour_sin"]     = np.sin(2*np.pi*local.hour/24)
    cal["hour_cos"]     = np.cos(2*np.pi*local.hour/24)
    # is_holiday / is_ramadan: PROMPT_1 adds via a Bangladesh holiday calendar.
    return cal
print("Calendar builder ready (weekend = Fri/Sat, Bangladesh convention).")

## Step 5 · Merge → master  — *PROMPT_1 completes*

In [ ]:
# === Step 5 · Merge -> master ===
# Continuous hourly UTC timeline over the config range. Left-join OpenAQ ground
# truth + ERA5 reanalysis + FIRMS (broadcast to hourly) + calendar + holiday/
# ramadan. Drop omaq_ammonia-like empty cols. Save parquet + csv.
from src.ingest.calendar_features import add_holiday_ramadan
from src.ingest.merge import build_master

master = build_master(
    aq_hourly=aq_hourly,
    met_hourly=met_hourly,
    fire_daily=fire_daily,
    add_calendar=add_calendar,                 # from Step 4 (pure)
    add_holiday_ramadan=add_holiday_ramadan,
    start=DR["start"], end=DR["end"],
    drop_columns=CFG.get("drop_columns", []),  # e.g. omaq_ammonia
)

print("master:", master.shape)
print("cols:", list(master.columns))
assert "openaq_pm25" in master.columns, "PM2.5 ground-truth column missing!"
assert not [c for c in master.columns if "co2" in c.lower() or "ammonia" in c.lower()], \
    "Forbidden CO2/ammonia column present."
_yr = master.assign(y=master["datetime_utc"].dt.year)
print("PM2.5 valid hours / year:")
print(_yr[_yr["openaq_pm25"].notna()].groupby("y").size())
print("Saved -> data/processed/dhaka_aq_master_rebuilt.parquet (+ .csv)")
master.head()

## Step 6 · Validate against the master
Skips loudly if files are missing; fails if you placed the wrong (gapped) file in data/reference/.

In [ ]:
!python -m pytest tests/test_against_reference.py -v